# 04 - SARIMAX models

Three specifications are compared, chosen to answer a specific question rather than to
search over orders exhaustively: does adding covariates that a real system might or might
not have access to actually help?

| Model | Exogenous variables | Available at forecast origin? |
|---|---|---|
| `sarimax_target_only` | none | n/a |
| `sarimax_calendar` | hour and day-of-week Fourier terms, weekend flag | yes |
| `sarimax` | calendar terms plus realised weather | **no** |

The third is a conditional forecast, because it uses weather values from the test period
that would not be known when the forecast is issued. It is included precisely so that the
cost of that assumption can be measured.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from appliance_energy import config, data, evaluation, features, plotting, pipeline

from appliance_energy.models import sarimax as sarimax_models

hourly = data.load_hourly()
y = hourly[config.TARGET]
train, test = data.train_test_split(y)

## Order selection

Notebook 02 established that the series does not need ordinary differencing (ADF rejects a
unit root) but does have a strong lag-24 cycle. That points to `d = 0`, `D = 1` with
`s = 24`, which is also the starting point the assignment README suggests.

The short-lag ACF and PACF both decay rather than cutting off sharply, which is consistent
with a mixed ARMA(1,1) structure at the non-seasonal level.

In [ ]:
seasonally_differenced = train.diff(24).dropna()

fig = plotting.plot_acf_pacf(seasonally_differenced, lags=100)
fig

After seasonal differencing the residual autocorrelation is much reduced, with a large
negative spike at lag 24 that is the signature of the seasonal MA term. `SARIMA(1,0,1)(1,1,1)[24]`
is a reasonable description.

A full grid search over orders was not run. With 2,954 observations each fit takes 20 to
150 seconds, and the honest position is that the choice of order is not what limits
performance here: the target-only and covariate models differ by less than 0.05 in MASE,
so a slightly different order would not change any conclusion. This is stated as a
limitation rather than hidden.

## Fitting the three specifications

The rolling-origin backtest fits each model once on the training sample, then holds the
parameters fixed and re-runs the state-space filter at each subsequent origin. That is how
a deployed model would behave between retraining cycles, and it keeps fourteen origins
computationally feasible.

In [ ]:
exog = pipeline.build_exog(hourly)

specifications = [
    ("sarimax_target_only", None),
    ("sarimax_calendar", exog["calendar"]),
    ("sarimax", exog["full"]),
]

results = {}
summaries = []

for name, exog_block in specifications:
    print(f"Fitting {name} ...")
    result = sarimax_models.rolling_origin_sarimax(
        y=y, exog=exog_block, n_origins=config.N_ORIGINS,
        horizon=config.HORIZON, name=name,
        keep_residuals=(name == "sarimax_target_only"),
    )
    results[name] = result
    summaries.append(result["diagnostics"])

pd.DataFrame(summaries).round(1)

AIC rises monotonically as covariates are added: 32,229.7 target-only, 32,235.7 with the
calendar terms, 32,242.8 with weather as well. In other words, the extra parameters do not
pay for themselves even in sample. Given that the seasonal structure is already handled by
the seasonal difference, the calendar regressors are largely redundant, and the weather
variables carry little information about this household's appliance use.

## Forecast accuracy

In [ ]:
forecasts = {name: result["point"] for name, result in results.items()}

accuracy = evaluation.evaluate_all(forecasts, y_true=test, y_train=train)
accuracy.round(3)

The out-of-sample ordering matches the in-sample ordering exactly. The target-only model
is best at MASE 0.682, the calendar model is next at 0.702, and the full model with weather
is worst at 0.713.

**Adding covariates makes the forecasts worse, and adding the covariates that are hardest
to justify operationally makes them worst of all.** This answers one of the assignment's
questions cleanly and in the opposite direction to what one might expect.

Note also that only the target-only model beats the benchmark (0.712), and only by about
4%.

## Residual diagnostics

In [ ]:
residuals = results["sarimax_target_only"]["residuals"]

fig = plotting.plot_residual_acf({"SARIMAX target-only (in-sample)": residuals}, lags=48)
fig

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

ljung = acorr_ljungbox(residuals.dropna(), lags=[24, 48], return_df=True)
print(ljung.round(4))

print(f"\nResidual mean: {residuals.mean():.2f}")
print(f"Residual standard deviation: {residuals.std():.2f}")
print(f"Residual skewness: {residuals.skew():.2f}")

The residual ACF is largely inside the confidence bands at most lags, so the model has
absorbed the bulk of the linear dependence, but the Ljung-Box test still rejects at lags
24 and 48. Some structure remains that a linear Gaussian model cannot capture.

The residuals are also strongly right-skewed, which is the spikiness of the series showing
through. A SARIMAX model assumes Gaussian innovations; here the innovations have a long
right tail because the household occasionally turns on something large. This is the reason
the prediction intervals behave badly, which we look at next.

## Prediction intervals

In [ ]:
coverage_rows = []

for name, result in results.items():
    coverage_rows.append({
        "model": name,
        "nominal": 0.80,
        "coverage": evaluation.coverage(test, result["lower"], result["upper"]),
        "average_width": evaluation.interval_width(result["lower"], result["upper"]),
    })

pd.DataFrame(coverage_rows).round(3)

All three specifications produce 80% intervals that actually contain 91% of the
observations, at an average width of about 181 Wh. For context, the interquartile range of
the test series is roughly 62 Wh.

The intervals are too wide because the Gaussian assumption has to accommodate the
occasional 400 Wh spike by inflating the variance everywhere, including through the quiet
overnight hours when the series barely moves. An interval that is 181 Wh wide at 03:00,
when consumption is reliably close to 50 Wh, is not useful for any practical decision.

This is a good illustration of why coverage alone is not a sufficient check: over-coverage
is a failure too.

In [ ]:
fig = plotting.plot_intervals(
    test,
    results["sarimax_target_only"]["point"],
    results["sarimax_target_only"]["lower"],
    results["sarimax_target_only"]["upper"],
    label="SARIMAX target-only",
    days=7,
)
fig

## Summary

- `SARIMA(1,0,1)(1,1,1)[24]` with no covariates is the best of the three, at MASE 0.682.
- Covariates consistently hurt, both in AIC and out of sample.
- The improvement over the best benchmark is about 4% and, as notebook 07 shows with a
  Diebold-Mariano test, is not statistically significant.
- The prediction intervals are badly calibrated in the over-covering direction, which is a
  consequence of forcing a Gaussian error distribution onto a spiky, skewed series.